# API Gateway를 AgentCore Gateway 대상으로 통합하기

## 개요

조직이 에이전틱 애플리케이션의 가능성을 탐색하면서, 엔터프라이즈 정책에 부합하는 안전한 방식으로 엔터프라이즈 데이터를 대규모 언어 모델(LLM) 호출 요청의 컨텍스트로 사용하는 과제를 계속 해결하고 있습니다. 이러한 상호 작용을 표준화하고 보호하기 위해 많은 조직이 에이전틱 애플리케이션을 데이터 소스 및 도구에 안전하게 연결하는 방법을 정의한 Model Context Protocol(MCP) 사양을 사용하고 있습니다.

MCP는 완전히 새로운 사용 사례에 유용하지만, 조직은 기존 API 자산을 에이전틱 시대에 활용하는 데에도 어려움을 겪고 있습니다. MCP로 기존 API를 래핑할 수는 있지만, MCP 요청을 RESTful API로 변환하고 전체 요청 흐름에서 보안을 유지하며 프로덕션 배포에 필요한 표준 관찰성을 적용하는 추가 작업이 필요합니다.

[Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)는 이제 [Amazon API Gateway](https://aws.amazon.com/api-gateway/)를 대상으로 지원하며, AgentCore Gateway(ACGW)로 전송된 MCP 요청을 API Gateway(APIGW)에 대한 RESTful 요청으로 변환합니다. 이제 기본 제공되는 보안 및 관찰성 기능과 함께 APIGW의 신규 및 기존 API 엔드포인트를 MCP를 통해 에이전틱 애플리케이션에 노출할 수 있습니다. 이 Notebook에서는 이 새로운 기능과 구현 방법을 다룹니다.

## 새로운 기능

AgentCore Gateway는 이미 Lambda 함수, OpenAPI 스키마, Smithy 모델, MCP 서버 등 다양한 대상 유형을 지원하며, 이제 API Gateway도 지원합니다.


![](Images/agent-core-gateway-targets.png)


**고객은 API Gateway를 사용하여 여러 애플리케이션의 백엔드를 연결하는 광범위한 API 에코시스템을 성공적으로 구축해 왔습니다.** 기업이 차세대 에이전틱 애플리케이션으로 발전함에 따라, 기존 API와 백엔드 도구를 AI 기반 시스템에 노출하여 기존 인프라와 최신 지능형 에이전트를 원활하게 통합하는 것은 자연스러운 변화입니다.

현재 고객은 APIGW API를 OpenAPI 3 사양으로 내보낸 다음 ACGW에 OpenAPI 대상으로 추가하는 수동 워크플로를 따릅니다. 이 통합은 APIGW와 ACGW 간 연결을 자동화하여 이 프로세스를 간소화하는 것을 목표로 합니다.


이 통합을 사용하면 고객이 내보내기/가져오기 프로세스를 직접 관리할 필요가 없습니다. ACGW에 새로운 API_GATEWAY 대상 유형이 추가됩니다. REST API 소유자는 몇 번의 콘솔 클릭이나 단일 CLI 명령으로 API를 ACGW 대상으로 추가하여 기존 REST API 메서드를 ACGW를 통해 MCP 도구로 노출할 수 있습니다. 그러면 API 사용자는 Model Context Protocol(MCP)을 통해 AI 에이전트를 이러한 REST API에 연결하고 AI 통합으로 워크플로를 강화할 수 있습니다. 이제 에이전틱 애플리케이션을 신규 또는 기존 APIGW API에 연결할 수 있습니다. 현재 ACGW와 APIGW 간 통합은 IAM 권한 부여와 API Key 권한 부여를 지원합니다.

![](Images/agent-core-apigw-target.png)

### 튜토리얼 세부 정보


| 정보                 | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형        | 대화형                                                    |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity                     |
| 에이전틱 프레임워크  | Strands Agents                                            |
| Gateway 대상 유형    | API Gateway                                               |
| 에이전트             | Strands                                                   |
| 인바운드 인증 IdP    | Amazon Cognito(다른 서비스도 사용 가능)                   |
| 아웃바운드 인증      | IAM 권한 부여 및 API Key                                  |
| LLM 모델             | Anthropic Claude Sonnet 4                                 |
| 튜토리얼 구성 요소   |AgentCore Gateway 대상을 통한 API Gateway 호출             |
| 튜토리얼 분야        | 산업 전반                                                 |
| 예제 난이도          | 쉬움                                                      |
| 사용 SDK             | boto3                                                     |

## 튜토리얼 아키텍처

이 튜토리얼은 다음과 같은 광범위한 엔터프라이즈 과제의 실용적인 예를 제공합니다. **차세대 에이전틱 애플리케이션을 위해 API Gateway API를 중앙 집중식 Gateway 아키텍처에 통합하는 방법**


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Jupyter notebook(Python 커널)
* uv
* AWS 자격 증명
* Amazon Cognito

In [ ]:
# 현재 디렉터리의 requirements 파일 또는 pyproject.toml 파일에서 설치
!pip3 install --force-reinstall -U -r requirements.txt --quiet

# 사용 중인 환경에 맞게 자유롭게 수정하세요. 다음 명령이 도움이 될 수 있습니다.
# !uv pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import os

# SageMaker notebook을 사용하지 않는 경우 AWS 자격 증명 설정
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-west-2")

In [ ]:
# utils 가져오기
import os
import sys
import importlib

# 현재 스크립트의 디렉터리 가져오기
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우(예: Jupyter) 대체 경로 사용

# utils.py는 Notebook과 같은 디렉터리에 있음
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# utils를 가져오고 변경 사항을 반영하도록 다시 로드
import utils

importlib.reload(utils)

# 로깅 설정
import logging

# Notebook 환경의 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

# 특정 로거 수준 설정
logging.getLogger("strands").setLevel(logging.INFO)

## 인바운드 및 아웃바운드 권한 부여를 위한 사전 요구 사항 설정

- 인바운드 권한 부여는 수신되는 사용자 요청을 인증합니다.
- 아웃바운드 권한 부여는 ACGW가 인증된 사용자를 대신하여 APIGW와 같은 Gateway 대상에 안전하게 연결하도록 지원합니다.

![](Images/agent-core-auth.png)
 
API Gateway를 대상으로 사용할 때 ACGW는 다음과 같은 아웃바운드 권한 부여 유형을 지원합니다.
-	권한 부여 없음(권장하지 않음) - 일부 대상 유형에서는 아웃바운드 권한 부여를 생략할 수 있습니다. 보안 수준이 낮은 이 옵션은 권장하지 않습니다.

-	IAM 기반 아웃바운드 권한 부여 - [Gateway 서비스 역할](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-prerequisites-permissions.html#gateway-service-role-permissions)을 사용하여 AWS Signature Version 4(Sig V4)로 Gateway 대상에 대한 액세스 권한을 부여합니다.

-	API Key - API Gateway에서 구성하고 AgentCore Identity에 설정한 API Key를 사용합니다. APIGW를 통해 생성된 API Key는 모니터링 및 제어에 도움이 되는 APIGW Usage Plan에 매핑됩니다. 자세한 내용은 이 [문서](https://docs.aws.amazon.com/apigateway/latest/developerguide/api-gateway-api-usage-plans.html)를 참조하세요.

IAM 기반 아웃바운드 권한 부여를 사용하려면 정책에 "execute-api:Invoke" 권한이 포함되어야 합니다.

#### ACGW가 수임할 IAM Role을 빠르게 생성해 보겠습니다. 
**APIGW를 호출할 때 IAM 기반 권한 부여에도 이 IAM Role을 사용합니다.**

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-agent-core-gateway-role-for-api-gateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

## Amazon Cognito를 사용하여 Gateway 인바운드 인증 구성

AgentCore Gateway로 수신되는 요청의 JWT 기반 인증을 처리하도록 Cognito User Pool을 생성합니다.

**생성되는 구성 요소:**
- 자격 증명 관리를 위한 User Pool
- OAuth 2.0 범위를 위한 Resource Server
- 프로그래밍 방식 액세스를 위한 Machine-to-Machine(M2M) Client


In [ ]:
# Cognito User Pool 생성
import os
import boto3

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {
        "ScopeName": "invoke",  # 'invoke'만 지정하면 resource_server_id/invoke 형식으로 구성됨
        "ScopeDescription": "Scope for invoking the agentcore gateway",
    },
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
    cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)

print(f"Client ID: {gw_client_id}")

# 검색 URL 가져오기
gw_cognito_discovery_url = (
    f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration"
)
print(gw_cognito_discovery_url)

## Gateway 생성
이제 AgentCore Gateway를 생성하겠습니다.

In [ ]:
# Cognito 권한 부여자를 사용하여 CreateGateway 실행. 이전 단계에서 생성한 Cognito User Pool 사용
import boto3

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            gw_client_id
        ],  # Client는 Cognito에 구성된 ClientId와 반드시 일치해야 함. 예: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": gw_cognito_discovery_url,
    }
}
create_response = gateway_client.create_gateway(
    name="sample-ac-gateway-apigw-demo",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM Role에는 Gateway 생성/목록 조회/가져오기/삭제 권한이 있어야 함
    protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26"], "searchType": "SEMANTIC"}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with API Gateway target",
)
print(create_response)
# GatewayTarget 생성에 사용할 GatewayID 가져오기
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

## Amazon API Gateway로 샘플 PetStore API 배포
### API 개요

IAM 및 API Key 권한 부여 패턴을 모두 보여 주는 샘플 PetStore REST API를 배포합니다.

**API 엔드포인트:**

| 엔드포인트 | 메서드 | 권한 부여 | 설명 |
|:---------|:-------|:--------------|:------------|
| `/pets` | GET | IAM (SigV4) | 사용 가능한 모든 반려동물 조회 |
| `/pets` | POST | IAM (SigV4) | 새 반려동물 추가 |
| `/pets/{petId}` | GET | IAM (SigV4) | ID로 반려동물 조회 |
| `/orders/{orderId}` | GET | API Key | 주문 세부 정보 조회 |

**생성되는 리소스:**
- 모의 통합이 포함된 REST API
- `/orders` 엔드포인트용 API Key
- 속도 제한이 적용된 Usage Plan
- `dev` 스테이지 배포

OpenAPI 사양은 [AgentCore_Sample_API-dev-oas30-apigateway.json](AgentCore_Sample_API-dev-oas30-apigateway.json)에서 확인할 수 있습니다.


In [ ]:
# API Gateway 확장이 포함된 OpenAPI 정의를 로드하고 API Gateway로 가져온 후 배포
# 통합 및 보안이 이미 구성된 OpenAPI 문서를 사용
api_result = utils.create_and_deploy_api_from_openapi_with_extensions(
    filename="AgentCore_Sample_API-dev-oas30-apigateway.json",
    stage_name="dev",
    description="Initial Deployment",
)

# 이후 셀에서 사용할 API 세부 정보 추출
api_id = api_result["api_id"]
api_key = api_result["api_key_value"]
invoke_url = api_result["invoke_url"]

print(f"\n{'=' * 70}")
print("API Details for Gateway Target Configuration")
print(f"{'=' * 70}")
print(f"API ID: {api_id}")
print(f"API Name: {api_result['api_name']}")
print(f"Stage: {api_result['stage_name']}")
print(f"API Key ID: {api_result['api_key_id']}")
print(f"Usage Plan ID: {api_result['usage_plan_id']}")
print(f"{'=' * 70}")

## API Gateway 배포 검증

모든 엔드포인트를 테스트하여 구성과 권한 부여가 올바른지 확인합니다.

**참고:** 테스트를 실행하기 전에 API Gateway 변경 사항이 가용 영역 전체에 전파되도록 10~15초 정도 기다리세요.


In [ ]:
test_results = utils.test_api_gateway_endpoints(
    invoke_url=api_result["invoke_url"],
    api_key=api_result["api_key_value"],
    region=os.environ.get("AWS_DEFAULT_REGION", "us-west-2"),
)

# 세부 결과 표시
print("\nDetailed Test Results:")
for endpoint, result in test_results.items():
    print(f"\n{endpoint}: {result['status']}")
    if "data" in result:
        print(f"  Data: {result['data']}")
    if "error" in result:
        print(f"  Error: {result['error']}")

## 생성된 API Key를 AgentCore Identity에서 안전하게 보호
다음 섹션에서는 AgentCore Identity에 새로운 API Key 자격 증명 공급자를 생성합니다. 이 공급자는 이후 `/orders` 엔드포인트용 AgentCore Gateway 대상을 생성할 때 사용됩니다. 

API Key는 AgentCore Identity를 통해 AWS Secrets Manager에 안전하게 저장됩니다.


In [ ]:
import boto3
import os

credentialProviderArn = ""
try:
    # Bedrock AgentCore Control 클라이언트 생성
    REGION = os.environ["AWS_DEFAULT_REGION"]
    bedrock_agent_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

    # AgentCore Identity에 API Key 자격 증명 공급자 생성
    credential_provider_response = bedrock_agent_client.create_api_key_credential_provider(
        name="sample-api-gateway-key-provider",
        apiKey=api_key,
        tags={"ProjectName": "AgentCore-APIGW-Sample-Application"},
    )

    # ARN을 변수에 저장
    secretArn = credential_provider_response["apiKeySecretArn"]["secretArn"]
    credentialProviderArn = credential_provider_response["credentialProviderArn"]
    print(f"Credential Provider Arn: {credentialProviderArn}")

except KeyError as e:
    print(f"Environment variable or response key missing: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

## Gateway 대상 생성
API Gateway 대상을 생성하려면 대상 구성에 다음 항목을 지정해야 합니다.
- **toolFilters**: REST API의 어떤 리소스를 ACGW의 도구로 노출할지 결정합니다. 필터는 filterPath에서 와일드카드도 지원합니다.
- **toolOverrides**(선택 사항): 사용자가 도구 이름과 설명을 재정의할 수 있도록 합니다. 명시적인 경로와 메서드를 지정해야 합니다. 지정하지 않으면 APIGW의 설명을 사용합니다.
- **restApiId**: API Gateway ID를 전달합니다. 
- **stage**: 배포된 API Gateway 스테이지 이름을 전달합니다.

![](Images/agent-core-apigw-target.png)

다음은 몇 가지 대상 구성 예제입니다.

### 예제 1:

이 구성은 “GET & POST /pets”와 “GET /pets/{petId}”를 Gateway에 노출하고 해당 도구 이름과 설명을 재정의합니다.

In [ ]:
{
    "mcp": {
        "apiGateway": {
            "restApiId": "<api-id>",
            "stage": "<stage>",
            "apiGatewayToolConfiguration": {
                "toolFilters": [
                    {"filterPath": "/pets", "methods": ["GET", "POST"]},
                    {"filterPath": "/pets/{petId}", "methods": ["GET"]},
                ],
                "toolOverrides": [
                    {
                        "name": "ListPets",
                        "path": "/pets",
                        "method": "GET",
                        "description": "Retrieves all the available Pets.",
                    },
                    {
                        "name": "AddPet",
                        "path": "/pets",
                        "method": "POST",
                        "description": "Add a new pet to the available Pets.",
                    },
                    {
                        "path": "/pets/{petId}",
                        "method": "GET",
                        "name": "GetPetById",
                        "description": "Retrieve a specific pet by its ID",
                    },
                ],
            },
        }
    }
}

### 예제 2
이 구성은 “GET /pets”뿐만 아니라 "GET /pets/{petId}" 또는 "/pets" 아래의 모든 항목을 노출합니다. toolOverrides를 지정하지 않았으므로 API Gateway의 리소스 설명을 사용합니다.

In [ ]:
{
    "mcp": {
        "apiGateway": {
            "restApiId": "<api-id>",
            "stage": "<stage>",
            "apiGatewayToolConfiguration": {"toolFilters": [{"filterPath": "/pets/*", "methods": ["GET"]}]},
        }
    }
}

## Gateway 대상 1 생성: IAM 권한 부여 엔드포인트

이 대상은 IAM 권한 부여를 사용하여 `/pets` 및 `/pets/{petId}` 엔드포인트를 노출합니다.


In [ ]:
import boto3

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

# IAM 자격 증명 공급자를 사용하여 '/pets' 및 '/pets/{petId}' 리소스용 Gateway 대상 생성
create_gateway_target_1_response = gateway_client.create_gateway_target(
    name="api-gateway-target-1",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "apiGateway": {
                "restApiId": api_id,
                "stage": "dev",
                "apiGatewayToolConfiguration": {
                    "toolFilters": [
                        {"filterPath": "/pets", "methods": ["GET", "POST"]},
                        {"filterPath": "/pets/{petId}", "methods": ["GET"]},
                    ],
                    "toolOverrides": [
                        {
                            "name": "List_Pets",
                            "path": "/pets",
                            "method": "GET",
                            "description": "Retrieves all the available Pets.",
                        },
                        {
                            "name": "Add_Pet",
                            "path": "/pets",
                            "method": "POST",
                            "description": "Add a new pet to the available Pets.",
                        },
                        {
                            "path": "/pets/{petId}",
                            "method": "GET",
                            "name": "GetPetById",
                            "description": "Retrieve a specific pet by its ID",
                        },
                    ],
                },
            }
        }
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
)
print(f"Create Gateway Target 1 Response: {create_gateway_target_1_response}")
gateway_target1_id = create_gateway_target_1_response["targetId"]

## Gateway 대상 2 생성: API Key 권한 부여 엔드포인트

이 대상은 앞에서 생성한 자격 증명 공급자의 API Key 권한 부여를 사용하여 `/orders/{orderId}` 엔드포인트를 노출합니다.


In [ ]:
# API Key 자격 증명 공급자를 사용하여 /orders/{orderId} 리소스용 두 번째 Gateway 대상 생성
create_gateway_target_2_response = gateway_client.create_gateway_target(
    name="api-gateway-target-2",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "apiGateway": {
                "restApiId": api_id,
                "stage": "dev",
                "apiGatewayToolConfiguration": {
                    "toolFilters": [{"filterPath": "/orders/{orderId}", "methods": ["GET"]}],
                    "toolOverrides": [
                        {
                            "path": "/orders/{orderId}",
                            "method": "GET",
                            "name": "GetOrderById",
                            "description": "Retrieve a specific order by its ID",
                        }
                    ],
                },
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": credentialProviderArn,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
)
print(f"Create Gateway Target 2 Response: {create_gateway_target_2_response}")
gateway_target2_id = create_gateway_target_2_response["targetId"]

## Gateway 대상 상태 확인

계속하기 전에 두 대상이 모두 `READY` 상태인지 확인합니다. 대상이 `FAILED` 상태이면 [get_gateway_target API](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agentcore-control/client/get_gateway_target.html)를 사용하여 실패 원인을 조사하세요.


In [ ]:
list_gateway_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID, maxResults=10)
for i, item in enumerate(list_gateway_targets_response["items"], 1):
    print(f"Target {i}:")
    print(f"  Target ID: {item['targetId']}")
    print(f"  Name: {item['name']}")
    print(f"  Status: {item['status']}")
    print()

## AI 에이전트와의 통합 테스트

Strands AI 프레임워크를 사용하여 다음과 같이 전체 통합을 테스트합니다.
1. Gateway에서 사용 가능한 도구 나열
2. 반려동물 관련 엔드포인트 호출(IAM 인증)
3. 주문 엔드포인트 호출(API Key 인증)


In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def get_token():
    token = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
    return token["access_token"]


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {get_token()}"})


client = MCPClient(create_streamable_http_transport)

## ~/.aws/credentials에 구성된 IAM 그룹/사용자에게 Bedrock 모델 액세스 권한이 있어야 함
yourmodel = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",  # 리전에 따라 model_id를 업데이트해야 할 수 있음
    temperature=0.7,
    max_tokens=500,  # 응답 길이 제한
)

with client:
    # listTools 호출
    tools = client.list_tools_sync()
    # 모델과 도구를 사용하여 에이전트 생성

    agent = Agent(model=yourmodel, tools=tools)
    ## 원하는 모델로 교체 가능

    print("\n" + "=" * 70)
    print("Test 1: List Available Tools")
    print("=" * 70)
    agent("List all tools available to you")

    print("\n" + "=" * 70)
    print("Test 2: List All Pets (IAM Auth)")
    print("=" * 70)
    agent("List all the available pets")

    print("\n" + "=" * 70)
    print("Test 3: Get Pet by ID (IAM Auth)")
    print("=" * 70)
    agent("Tell me about the pet with petId 3")

    print("\n" + "=" * 70)
    print("Test 4: Get Order Details (API Key Auth)")
    print("=" * 70)
    agent("When will my order be delivered? My order id is 2")

## 리소스 정리

다음 셀을 실행하여 이 튜토리얼에서 생성한 모든 리소스를 제거합니다.

**삭제할 리소스:**
1. AgentCore Gateway 및 대상
2. AgentCore Identity 자격 증명 공급자
3. API Gateway REST API, API Key 및 Usage Plan
4. Amazon Cognito User Pool 및 클라이언트
5. IAM 역할 


In [ ]:
# AgentCore Gateway 및 모든 대상 삭제
print("Step 1: Cleaning up AgentCore Gateway resources...")
agentcore_cleanup = utils.delete_agentcore_gateway_and_targets(gateway_id=gatewayID, region=REGION)

# AgentCore Identity 자격 증명 공급자 삭제
print("\nStep 2: Cleaning up AgentCore Identity Credential Provider...")
credential_cleanup = utils.delete_agentcore_credential_provider(
    credential_provider_arn=credentialProviderArn, region=REGION
)

# API Gateway 및 모든 관련 리소스 삭제
print("\nStep 3: Cleaning up API Gateway resources...")
api_cleanup = utils.delete_api_gateway_and_resources(
    api_id=api_id,
    api_key_id=api_result.get("api_key_id"),
    usage_plan_id=api_result.get("usage_plan_id"),
)

# Cognito User Pool 삭제
print("\nStep 4: Cleaning up Cognito User Pool...")
cognito_cleanup = utils.delete_cognito_user_pool(user_pool_name="sample-agentcore-gateway-pool", region=REGION)

# IAM Role 삭제
print("\nStep 5: Cleaning up IAM Role...")
iam_cleanup = utils.delete_iam_role(role_name="agentcore-sample-agent-core-gateway-role-role")

# 전체 요약
print("\n" + "=" * 70)
print("Overall Cleanup Summary")
print("=" * 70)
print(f"AgentCore Gateway Deleted: {'✓' if agentcore_cleanup['gateway_deleted'] else '✗'}")
print(f"AgentCore Targets Deleted: {len(agentcore_cleanup['targets_deleted'])}")
print(f"Credential Provider Deleted: {'✓' if credential_cleanup['credential_provider_deleted'] else '✗'}")
print(f"Cognito User Pool Deleted: {'✓' if cognito_cleanup['user_pool_deleted'] else '✗'}")
print(f"Cognito Clients Deleted: {len(cognito_cleanup['clients_deleted'])}")
print(f"IAM Role Deleted: {'✓' if iam_cleanup['role_deleted'] else '✗'}")
print(f"IAM Policies Deleted: {len(iam_cleanup['policies_deleted'])}")
print(f"API Gateway Deleted: {'✓' if api_cleanup['api_deleted'] else '✗'}")
print(f"API Key Deleted: {'✓' if api_cleanup.get('api_key_deleted') else '✗'}")
print(f"Usage Plan Deleted: {'✓' if api_cleanup.get('usage_plan_deleted') else '✗'}")
print("=" * 70)

## 요약
AgentCore Gateway와 API Gateway 간의 새로운 통합을 성공적으로 테스트했습니다. 이 통합을 통해 기존 REST API를 차세대 에이전틱 애플리케이션을 위한 MCP 호환 엔드포인트로 노출할 수 있습니다. 